In [3]:
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

In [4]:
!pip install pyspark

In [18]:
from pyspark.sql import SparkSession

# 1. Khởi tạo phiên làm việc Spark
spark = SparkSession.builder \
    .appName("NYC_Taxi_HackData") \
    .getOrCreate()

# 2. Đọc dữ liệu
# QUAN TRỌNG: Bạn thay phần trong ngoặc kép bằng đường dẫn thực tế tới THƯ MỤC GỐC chứa data trên Drive của bạn
# Ví dụ: '/content/drive/MyDrive/Hackathon_Data/Taxi_Dataset/'
dataset_path = "/content/drive/MyDrive/BigData/timeseries_taxi.parquet/PULocationID=100"

df = spark.read.parquet(dataset_path)

In [ ]:
# In ra tên cột và kiểu dữ liệu (Schema)
print("=== CẤU TRÚC CÁC CỘT (SCHEMA) ===")
df.printSchema()

# In thử 5 dòng đầu tiên để xem giá trị thực tế bên trong
print("\n=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===")
df.show(5, truncate=False)

# In tổng số dòng hiện có trong tập dữ liệu
print(f"\nTổng số dòng: {df.count()}")

=== CẤU TRÚC CÁC CỘT (SCHEMA) ===
root
 |-- timestamp: timestamp (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- demand_count: long (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_duration: double (nullable = true)
 |-- total_revenue: double (nullable = true)


=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===
+-------------------+-----------+---------+----------+-----+----+------------+------------+--------+------------------+-------------+
|timestamp          |pickup_hour|dayofweek|is_weekend|month|year|demand_count|avg_distance|avg_fare|avg_duration      |total_revenue|
+-------------------+-----------+---------+----------+-----+----+------------+------------+--------+------------------+-------------+
|2022-01-01 15:00:00|15         |7        |1         |1    |20

In [19]:
df = df.toPandas()


In [20]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')
print(f"Đã đọc xong dữ liệu! Tổng số dòng: {len(df)}")


Đã đọc xong dữ liệu! Tổng số dòng: 7260


In [21]:
df = df.resample('h').asfreq()

cols_to_zero = ['demand_count', 'avg_distance', 'avg_fare', 'avg_duration', 'total_revenue']
df[cols_to_zero] = df[cols_to_zero].fillna(0)

df['pickup_hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek + 1
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 6 else 0)

features = ['demand_count', 'total_revenue', 'pickup_hour', 'dayofweek', 'is_weekend']
data = df[features].values

In [22]:
from sklearn.preprocessing import MinMaxScaler

# Scale toàn bộ input
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Ta cần một scaler riêng biệt chỉ cho cột demand_count (cột index 0)
# để xíu nữa giải mã (inverse_transform) kết quả dự đoán về lại số thực tế
target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler.fit(df[['demand_count']])

MinMaxScaler()

In [23]:
def create_dataset(dataset, time_steps=24):
    X, y = [], []
    for i in range(len(dataset) - time_steps):
        X.append(dataset[i:(i + time_steps), :])
        y.append(dataset[i + time_steps, 0])
    return np.array(X), np.array(y)

time_steps = 24
X, y = create_dataset(data, time_steps)

train_size = int(len(X) * 0.8)
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

print(f"Shape của X_train (LSTM): {X_train.shape} -> (samples, time_steps, features)")

Shape của X_train (LSTM): (7584, 24, 5) -> (samples, time_steps, features)


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng: {device}")

X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.FloatTensor(y_train).view(-1, 1).to(device) # view(-1,1) để biến [n] thành [n, 1]

X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.FloatTensor(y_test).view(-1, 1).to(device)

# 2. Tạo DataLoader để chia batch
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

Đang sử dụng: cpu


In [32]:
# 3. Định nghĩa mô hình LSTM
class TaxiLSTM(nn.Module):
    def __init__(self, input_size):
        super(TaxiLSTM, self).__init__()

        # Lớp LSTM 1 (64 units)
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=64, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)

        # Lớp LSTM 2 (32 units)
        self.lstm2 = nn.LSTM(input_size=64, hidden_size=32, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)

        # Lớp Fully Connected (Linear) để xuất ra 1 giá trị dự đoán
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        # Truyền qua LSTM 1
        out, _ = self.lstm1(x)
        out = self.dropout1(out)

        # Truyền qua LSTM 2
        out, _ = self.lstm2(out)
        out = self.dropout2(out)

        # Chỉ lấy hidden state của bước thời gian cuối cùng để dự báo
        out = out[:, -1, :]

        # Truyền qua lớp Linear
        out = self.fc(out)
        return out

# Khởi tạo mô hình
input_features = X_train.shape[2]
model = TaxiLSTM(input_size=input_features).to(device)

# 4. Định nghĩa Hàm mất mát (Loss) và Bộ tối ưu (Optimizer)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [33]:
# 5. Vòng lặp huấn luyện (Training Loop)
epochs = 100

print("Bắt đầu huấn luyện mô hình LSTM bằng PyTorch...")
for epoch in range(epochs):
    model.train() # Đặt mô hình ở chế độ huấn luyện
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        # Reset gradient
        optimizer.zero_grad()

        # Feed Forward (Dự đoán)
        predictions = model(batch_X)

        # Tính Loss
        loss = criterion(predictions, batch_y)

        # Backpropagation (Lan truyền ngược)
        loss.backward()

        # Cập nhật trọng số
        optimizer.step()

        epoch_loss += loss.item()

    # In ra tiến trình sau mỗi epoch
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss (MSE): {avg_loss:.4f}")

print("Huấn luyện hoàn tất!")

Bắt đầu huấn luyện mô hình LSTM bằng PyTorch...
Epoch [1/100], Loss (MSE): 3779.3860
Epoch [2/100], Loss (MSE): 3188.2264
Epoch [3/100], Loss (MSE): 2735.0578
Epoch [4/100], Loss (MSE): 2349.5206
Epoch [5/100], Loss (MSE): 2039.0992
Epoch [6/100], Loss (MSE): 1778.9630
Epoch [7/100], Loss (MSE): 1540.4607
Epoch [8/100], Loss (MSE): 1306.3715
Epoch [9/100], Loss (MSE): 1123.4390
Epoch [10/100], Loss (MSE): 962.3047
Epoch [11/100], Loss (MSE): 838.1715
Epoch [12/100], Loss (MSE): 724.2559
Epoch [13/100], Loss (MSE): 635.4419
Epoch [14/100], Loss (MSE): 568.3768
Epoch [15/100], Loss (MSE): 513.9686
Epoch [16/100], Loss (MSE): 473.7141
Epoch [17/100], Loss (MSE): 435.2726
Epoch [18/100], Loss (MSE): 400.3931
Epoch [19/100], Loss (MSE): 367.7144
Epoch [20/100], Loss (MSE): 366.9718
Epoch [21/100], Loss (MSE): 361.8305
Epoch [22/100], Loss (MSE): 350.0488
Epoch [23/100], Loss (MSE): 313.4431
Epoch [24/100], Loss (MSE): 320.3426
Epoch [25/100], Loss (MSE): 327.6175
Epoch [26/100], Loss (MSE):

In [36]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

# 1. Dự báo với LSTM
model.eval() # Bắt buộc: Tắt Dropout và đưa mô hình về chế độ test
with torch.no_grad():
    # Đưa tập test vào mô hình (nhớ dùng X_test_tensor đã tạo ở phần trước)
    lstm_pred_tensor = model(X_test_tensor)
    # Đưa kết quả từ GPU/Tensor về lại Numpy array trên CPU
    lstm_pred_scaled = lstm_pred_tensor.cpu().numpy()

# 2. Giải mã (Inverse Transform) để đưa giá trị (0-1) về lại số lượng chuyến xe thực tế
y_test_actual = target_scaler.inverse_transform(y_test.reshape(-1, 1))
lstm_pred_actual = target_scaler.inverse_transform(lstm_pred_scaled)

# 3. Hàm in kết quả đánh giá (Dùng chung cho cả 3 mô hình)
def print_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- Kết quả của {model_name} ---")
    print(f"MAE  : {mae:.2f} (chuyến)")
    print(f"RMSE : {rmse:.2f} (chuyến)")
    print(f"R2   : {r2:.4f}\n")

print_metrics(y_test_actual, lstm_pred_actual, "LSTM (PyTorch)")

--- Kết quả của LSTM (PyTorch) ---
MAE  : 2355.88 (chuyến)
RMSE : 3133.41 (chuyến)
R2   : 0.7499



In [38]:
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

# 1. Làm phẳng dữ liệu 3D thành 2D
X_train_ml = X_train.reshape(X_train.shape[0], -1)
X_test_ml = X_test.reshape(X_test.shape[0], -1)

print("Kích thước dữ liệu cho ML:", X_train_ml.shape)

# ==========================================
# 2. RANDOM FOREST
# ==========================================
print("\nBắt đầu train Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_ml, y_train)

# Dự báo và giải mã
rf_pred_scaled = rf_model.predict(X_test_ml).reshape(-1, 1)
rf_pred_actual = target_scaler.inverse_transform(rf_pred_scaled)

print_metrics(y_test_actual, rf_pred_actual, "Random Forest")


# ==========================================
# 3. XGBOOST
# ==========================================
print("Bắt đầu train XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train_ml, y_train)

# Dự báo và giải mã
xgb_pred_scaled = xgb_model.predict(X_test_ml).reshape(-1, 1)
xgb_pred_actual = target_scaler.inverse_transform(xgb_pred_scaled)

print_metrics(y_test_actual, xgb_pred_actual, "XGBoost")

Kích thước dữ liệu cho ML: (7584, 120)

Bắt đầu train Random Forest...
--- Kết quả của Random Forest ---
MAE  : 1937.71 (chuyến)
RMSE : 2568.50 (chuyến)
R2   : 0.8320

Bắt đầu train XGBoost...
--- Kết quả của XGBoost ---
MAE  : 1885.38 (chuyến)
RMSE : 2500.64 (chuyến)
R2   : 0.8407



In [37]:
import plotly.graph_objects as go

# Chọn vẽ khoảng 300 giờ đầu tiên (khoảng hơn 10 ngày) của tập test để biểu đồ dễ nhìn
plot_range = 300

# Làm phẳng mảng dữ liệu (đảm bảo là mảng 1 chiều)
actual_data = y_test_actual[:plot_range].flatten()
lstm_data = lstm_pred_actual[:plot_range].flatten()

# Khởi tạo biểu đồ
fig = go.Figure()

# Thêm đường dữ liệu thực tế
fig.add_trace(go.Scatter(
    y=actual_data,
    mode='lines',
    name='Thực tế (Actual Demand)',
    line=dict(color='rgba(0, 0, 0, 0.8)', width=2)
))

# Thêm đường dự báo của LSTM
fig.add_trace(go.Scatter(
    y=lstm_data,
    mode='lines',
    name='Dự báo LSTM (PyTorch)',
    line=dict(color='rgba(31, 119, 180, 0.9)', width=2, dash='dot')
))

# Làm đẹp biểu đồ
fig.update_layout(
    title=dict(
        text='So sánh Dự báo Nhu cầu Di chuyển Taxi NYC (Tập Test)',
        font=dict(size=20)
    ),
    xaxis_title='Khung thời gian (Giờ)',
    yaxis_title='Số lượng chuyến xe (Demand)',
    template='plotly_white',
    hovermode='x unified', # Hiển thị popup so sánh 2 đường khi di chuột
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

# Hiển thị
fig.show()

In [40]:
import torch
import joblib
import os

# Tạo một thư mục trên Drive để chứa model cho gọn gàng
save_dir = '/content/drive/MyDrive/BigData/Model/'
os.makedirs(save_dir, exist_ok=True)

In [41]:
# 1. Lưu Scalers
joblib.dump(scaler, save_dir + 'feature_scaler.pkl')
joblib.dump(target_scaler, save_dir + 'target_scaler.pkl')
print("Đã lưu thành công các Scalers!")

# 2. Lưu Random Forest
joblib.dump(rf_model, save_dir + 'random_forest_model.pkl')
print("Đã lưu thành công Random Forest!")

# 3. Lưu XGBoost
xgb_model.save_model(save_dir + 'xgboost_model.json')
print("Đã lưu thành công XGBoost!")

# 4. Lưu LSTM (Lưu trọng số)
torch.save(model.state_dict(), save_dir + 'lstm_model.pth')
print("Đã lưu thành công LSTM!")

Đã lưu thành công các Scalers!
Đã lưu thành công Random Forest!
Đã lưu thành công XGBoost!
Đã lưu thành công LSTM!
